# Commerce synthetic scan OCR CPU smoke check

Run one OCR engine on the two-page image-only Commerce synthetic fixture with Kaggle GPU disabled. This checks CPU installation and evaluator wiring only; it is not a representative accuracy benchmark. Attach the private asset bundle produced by `tools/processing/package_ocr_benchmark.py`.

The bundle contains the evaluator, visual-transcription labels, and a watermarked synthetic PDF; it contains no credentials or user uploads. The sample prose and table values are invented, English-only, and must not be treated as Commerce facts or policy. Record engine version, CPU architecture/count, page latency, and memory when comparing repeat runs.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

input_root = Path("/kaggle/input")
for bundle in [*input_root.glob("*/benchmark_project.zip"), *input_root.glob("datasets/*/*/benchmark_project.zip")]:
    extracted = Path("/kaggle/working/benchmark-assets/project")
    shutil.rmtree(extracted.parent, ignore_errors=True)
    shutil.unpack_archive(str(bundle), str(extracted.parent))
project_candidates = {
    *input_root.glob("*/project"),
    *input_root.glob("datasets/*/*/project"),
    *Path("/kaggle/working/benchmark-assets").glob("project"),
    *(entry.parent.parent.parent for entry in input_root.rglob("evaluate_ocr.py") if entry.parent.name == "processing" and entry.parent.parent.name == "scripts"),
}
assets_candidates = sorted(
    project for project in project_candidates
    if (project / "scripts" / "processing" / "evaluate_ocr.py").is_file()
)
if len(assets_candidates) != 1:
    raise FileNotFoundError(
        "Expected one attached Commerce OCR smoke assets/project; "
        f"found: {assets_candidates}"
    )
assets = assets_candidates[0]
repo = Path("/kaggle/working/enterprise-knowledge-ai-agent")
if repo.exists():
    shutil.rmtree(repo)
shutil.copytree(assets, repo)
%cd /kaggle/working/enterprise-knowledge-ai-agent


In [ ]:
ENGINE = "tesseract"  # Change to paddleocr, easyocr, or surya for separate kernels.
DPI = 180
TESSERACT_PSM = 3  # Use 11 for the paired sparse-text layout comparison.
DATASET_PATH = "data/evaluation/ocr/commerce_scan_ground_truth.json"
DATASET_TAG = "commerce-synthetic-smoke-v1"


In [ ]:
import subprocess, sys

# Install one engine only, avoiding incompatible OCR framework combinations.
subprocess.run([sys.executable, "-m", "pip", "install", "PyMuPDF==1.27.2.3", "Pillow==12.1.0"], check=True)
if ENGINE == "paddleocr":
    subprocess.run([sys.executable, "-m", "pip", "--default-timeout=300", "--retries=5", "install", "paddlepaddle==3.3.0", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cpu/"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "paddleocr==3.7.0"], check=True)
elif ENGINE == "easyocr":
    subprocess.run([sys.executable, "-m", "pip", "install", "easyocr==1.7.2"], check=True)
elif ENGINE == "surya":
    subprocess.run([sys.executable, "-m", "pip", "install", "surya-ocr==0.17.1", "transformers==4.57.3"], check=True)
elif ENGINE == "tesseract":
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-vie"], check=True)
else:
    raise ValueError(f"Unknown ENGINE: {ENGINE}")


In [ ]:
import os

os.environ["HF_HOME"] = "/tmp/ocr-model-cache/huggingface"
os.environ["TORCH_HOME"] = "/tmp/ocr-model-cache/torch"
if ENGINE == "surya":
    # Limit per-page CPU inference memory for this paired run.
    os.environ["DETECTOR_BATCH_SIZE"] = "4"
    os.environ["RECOGNITION_BATCH_SIZE"] = "16"
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
import torch
cuda_available = torch.cuda.is_available()
print("torch", torch.__version__, "CUDA", cuda_available)
if cuda_available:
    raise RuntimeError("This CPU smoke check requires a CPU-only Kaggle runtime.")
if ENGINE == "paddleocr":
    import paddle
    paddle_device = paddle.device.get_device()
    print("paddle", paddle.__version__, "CUDA build", paddle.is_compiled_with_cuda(), "device", paddle_device)
    if paddle_device != "cpu":
        raise RuntimeError("The CPU smoke check requires PaddleOCR CPU execution.")


In [ ]:
import json, os, platform
dataset_path = Path(DATASET_PATH)
if not dataset_path.is_file():
    raise FileNotFoundError(f"Missing OCR labels: {dataset_path}")
dataset = json.loads(dataset_path.read_text(encoding="utf-8"))
for source in sorted({page["file"] for page in dataset["pages"]}):
    if not Path(source).is_file():
        raise FileNotFoundError(f"Missing benchmark source PDF: {source}")
output = Path("/kaggle/working") / f"commerce-ocr-{DATASET_TAG}-{ENGINE}.json"
command = [sys.executable, "tools/processing/evaluate_ocr.py", "--dataset", DATASET_PATH, "--engines", ENGINE, "--dpi", str(DPI), "--output", str(output)]
if ENGINE == "tesseract":
    command.extend(["--tesseract-psm", str(TESSERACT_PSM)])
subprocess.run(command, check=True)
report = json.loads(output.read_text(encoding="utf-8"))
cpu_model = None
if Path("/proc/cpuinfo").is_file():
    for line in Path("/proc/cpuinfo").read_text(errors="replace").splitlines():
        if line.lower().startswith(("model name", "hardware")):
            cpu_model = line.split(":", 1)[-1].strip()
            if cpu_model:
                break
if not cpu_model:
    candidate = platform.processor()
    cpu_model = candidate if candidate and candidate != platform.machine() else None
report["environment"]["cpu_model"] = cpu_model or "not exposed by runtime"
report["environment"]["cpu_architecture"] = platform.machine()
report["environment"]["logical_cpus"] = os.cpu_count()
output.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Download the result from Kaggle Outputs: {output}")


## Limitations

This paired run exercises four engines on a clean two-page synthetic fixture. Exact-match metrics only validate this test path; they do not represent scan quality in production documents. CPU host variation also makes throughput comparisons approximate.